# V7 Trainable Quantum Rerun (Clean Colab Notebook)

This notebook is the clean Colab path for the April 2026 V7 confirmatory rerun.

Current purpose:
- produce a fresh artifact-backed V7 run with a top-level `experiments/*.json` row
- keep Drive-backed checkpoints
- allow resume after disconnection
- continue beyond an early target hit by rerunning with a higher `--target`
- copy the JSON row and experiment directory back to Drive


## 1. Runtime Check

Use an L4 if available. A100 is also acceptable.


In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone or Refresh Repo

If the folder already exists, this cell refreshes it.


In [ ]:
import os

repo_path = '/content/Quanvolutional-Neural-Network'
if not os.path.exists(repo_path):
    !git clone https://github.com/necatiincekara/Quanvolutional-Neural-Network.git /content/Quanvolutional-Neural-Network
else:
    %cd /content/Quanvolutional-Neural-Network
    !git pull

%cd /content/Quanvolutional-Neural-Network
!git branch
!git log --oneline -n 3


## 4. Install Dependencies


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!pip install -r requirements.txt


## 5. Stage Dataset To Local Colab Disk

Drive remains the source of truth; local disk is used for speed.


In [ ]:
!mkdir -p /content/local_data/train /content/local_data/test
!rsync -a /content/drive/MyDrive/set/train/ /content/local_data/train/
!rsync -a /content/drive/MyDrive/set/test/ /content/local_data/test/
!find /content/local_data/train -type f | wc -l
!find /content/local_data/test -type f | wc -l


## 6. Sanity Check CLI

The updated entrypoint should support `--resume`, `--drive-backup-path`, `--train-path`, `--test-path`, and `--result-json`.


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!python train_v7.py --help


## 7. Clean Fresh Confirmatory Run

Use this first. It intentionally clears local V7 checkpoints so the run starts from scratch.


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!mkdir -p /content/drive/MyDrive/quanv_results/v7_clean_20260427/experiments
!rm -f models/checkpoint_latest_v7.pth models/best_v7_model.pth
!python train_v7.py \
  --circuit data_reuploading \
  --epochs 10 \
  --target 90 \
  --drive-backup-path /content/drive/MyDrive/quanv_results/v7_clean_20260427 \
  --train-path /content/local_data/train \
  --test-path /content/local_data/test \
  --result-json experiments/v7_trainable_quantum_clean_20260427_l4.json


## 8. Continue Past Early Target Hit

If training stops early because the target was reached, rerun with a higher target and `--resume`.
Example below continues the same clean run toward the full 10-epoch budget.


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!python train_v7.py \
  --circuit data_reuploading \
  --epochs 10 \
  --target 90 \
  --resume \
  --drive-backup-path /content/drive/MyDrive/quanv_results/v7_clean_20260427 \
  --train-path /content/local_data/train \
  --test-path /content/local_data/test \
  --result-json experiments/v7_trainable_quantum_clean_20260427_l4.json


## 9. Inspect Saved Artifacts


In [ ]:
!RUN_DIR=/content/drive/MyDrive/quanv_results/v7_clean_20260427; \
  cp experiments/v7_trainable_quantum_clean_20260427_l4.json "$RUN_DIR/"; \
  rsync -a experiments/v7_data_reuploading_*/ "$RUN_DIR/experiments/"; \
  ls -lah models; \
  ls -lah "$RUN_DIR"; \
  ls -lah "$RUN_DIR/experiments"; \
  python -m json.tool "$RUN_DIR/v7_trainable_quantum_clean_20260427_l4.json" | head -80
